# Notebook 04: Transfer Learning — EfficientNet-B0

**Student:** Okidi Patrovas Gabriel | 2025/HD07/26018U  
**Institution:** Makerere University, Kampala, Uganda  
**Course:** MSB7216 — Deep Learning for Health Data

## Overview
This notebook fine-tunes EfficientNet-B0 pretrained on ImageNet for five-class classification on the LC25000 dataset. EfficientNet-B0 was chosen because it achieves strong accuracy with a compact architecture, making it well suited for deployment. We use a two-phase training strategy: first training only the classifier head, then unfreezing the full network for fine-tuning.

## Objectives
1. Load the same train/val/test splits used in Notebook 03.
2. Load images from local Colab storage for fast GPU training.
3. Build EfficientNet-B0 with a replaced classifier head.
4. Train in two phases: head-only warmup, then full fine-tuning.
5. Plot training and validation curves.
6. Evaluate on the test set with all metrics.
7. Plot confusion matrix and ROC curves.
8. Document the overfitting gap and compare to the baseline.

In [ ]:
# ── Cell 1: Install and Import ────────────────────────────────────────────────
!pip install torch torchvision --quiet

import os, random, warnings, json, copy, time, shutil
import concurrent.futures
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU not available. Go to Runtime > Change runtime type > GPU')

In [ ]:
# ── Cell 2: Mount Drive and Load Splits ───────────────────────────────────────
drive.mount('/content/drive', force_remount=False)

BASE_DIR    = Path('/content/drive/MyDrive/lung-colon-cancer-histopathology')
FIGURES_DIR = BASE_DIR / 'figures'
MODELS_DIR  = BASE_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open(BASE_DIR / 'data' / 'dataset_splits.json', 'r') as f:
    split_data = json.load(f)

train_paths  = split_data['train_paths']
val_paths    = split_data['val_paths']
test_paths   = split_data['test_paths']
train_labels = split_data['train_labels']
val_labels   = split_data['val_labels']
test_labels  = split_data['test_labels']
CLASS_NAMES  = split_data['class_names']

print(f'Train : {len(train_paths):,} | Val : {len(val_paths):,} | Test : {len(test_paths):,}')
print(f'Classes : {CLASS_NAMES}')

In [ ]:
# ── Cell 3: Copy Dataset to Local Colab Storage ───────────────────────────────
# If already copied in this session, this cell skips automatically.
# Uses parallel threads for fast copying.

LOCAL_DATA = Path('/content/local_data')

# Check how many files already exist locally
existing = len(list(LOCAL_DATA.rglob('*.jpeg'))) if LOCAL_DATA.exists() else 0
total_needed = len(train_paths) + len(val_paths) + len(test_paths)

if existing >= total_needed:
    print(f'Local dataset already complete: {existing} files. Skipping copy.')
else:
    print(f'Found {existing}/{total_needed} files locally. Copying missing files...')

    all_paths_combined  = train_paths + val_paths + test_paths
    all_labels_combined = train_labels + val_labels + test_labels

    copy_tasks = []
    for src_path, label in zip(all_paths_combined, all_labels_combined):
        class_name = CLASS_NAMES[label]
        dst_dir    = LOCAL_DATA / class_name
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst_path   = dst_dir / Path(src_path).name
        if not dst_path.exists():
            copy_tasks.append((str(src_path), str(dst_path)))

    if copy_tasks:
        print(f'Copying {len(copy_tasks)} files using parallel threads...')
        start = time.time()

        def copy_file(args):
            src, dst = args
            shutil.copy2(src, dst)
            return dst

        with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
            list(executor.map(copy_file, copy_tasks))

        elapsed = time.time() - start
        total   = len(list(LOCAL_DATA.rglob('*.jpeg')))
        print(f'Done. {total} files ready in {elapsed:.0f} seconds.')
    else:
        print('All files already present.')

# Update all paths to point to local storage
def localise_paths(paths, labels):
    return [
        str(LOCAL_DATA / CLASS_NAMES[l] / Path(p).name)
        for p, l in zip(paths, labels)
    ]

train_paths_local = localise_paths(train_paths, train_labels)
val_paths_local   = localise_paths(val_paths,   val_labels)
test_paths_local  = localise_paths(test_paths,  test_labels)

print(f'Train paths : {len(train_paths_local)} -> {train_paths_local[0]}')
print(f'Val paths   : {len(val_paths_local)}')
print(f'Test paths  : {len(test_paths_local)}')

In [ ]:
# ── Cell 4: Build DataLoaders from Local Storage ──────────────────────────────

IMAGE_SIZE    = 224
BATCH_SIZE    = 32
NUM_WORKERS   = 2
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class LC25000Dataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Use local paths — NOT Drive paths
train_loader = DataLoader(
    LC25000Dataset(train_paths_local, train_labels, train_transform),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    LC25000Dataset(val_paths_local, val_labels, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    LC25000Dataset(test_paths_local, test_labels, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')
print(f'Reading from  : {train_loader.dataset.image_paths[0]}')

In [ ]:
# ── Cell 5: Build EfficientNet-B0 ─────────────────────────────────────────────

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Freeze all backbone layers for warmup phase
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, 5),
)

model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'EfficientNet-B0 loaded with ImageNet weights.')
print(f'Trainable parameters (warmup) : {trainable:,} / {total:,}')
print(f'Model device                  : {next(model.parameters()).device}')

In [ ]:
# ── Cell 6: Training Setup and Helper Function ────────────────────────────────

CHECKPOINT = str(MODELS_DIR / 'efficientnet_b0_best.pth')
RESUME_PATH = str(MODELS_DIR / 'efficientnet_b0_checkpoint.pth')
HISTORY_PATH = str(MODELS_DIR / 'efficientnet_b0_history.json')

criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += outputs.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total

print('Training setup ready.')
print(f'Checkpoint path : {CHECKPOINT}')

In [ ]:
# ── Cell 7: Phase 1 — Warmup (Head Only, 5 Epochs) ───────────────────────────

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

# Check if we should skip warmup due to existing checkpoint
warmup_done = os.path.exists(RESUME_PATH)

if not warmup_done:
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
    )
    print('Phase 1: Warmup (head only, 5 epochs)')
    for epoch in range(1, 6):
        start = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = run_epoch(model, val_loader,   criterion)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        elapsed = time.time() - start
        print(f'  Epoch {epoch}/5 | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
              f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f} | Time: {elapsed:.1f}s')
    print('Warmup complete.')
else:
    print('Checkpoint found. Skipping warmup and loading from checkpoint.')

In [ ]:
# ── Cell 8: Phase 2 — Full Fine-Tuning with Resumable Checkpointing ───────────

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'All layers unfrozen. Trainable parameters : {trainable:,}')

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

NUM_EPOCHS        = 20
PATIENCE          = 5
best_val_loss     = float('inf')
best_weights      = copy.deepcopy(model.state_dict())
epochs_no_improve = 0
START_EPOCH       = 1

# Resume from checkpoint if it exists
if os.path.exists(RESUME_PATH) and os.path.exists(HISTORY_PATH):
    print('Resuming from saved checkpoint...')
    checkpoint    = torch.load(RESUME_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    best_val_loss = checkpoint['best_val_loss']
    START_EPOCH   = checkpoint['epoch'] + 1
    with open(HISTORY_PATH, 'r') as f:
        history = json.load(f)
    print(f'Resumed from epoch {START_EPOCH - 1}.')
    print(f'Best val loss so far : {best_val_loss:.4f}')
else:
    print('Starting Phase 2 fine-tuning from scratch.')

print(f'\nPhase 2: Full fine-tuning (max {NUM_EPOCHS} epochs, patience {PATIENCE})')

for epoch in range(START_EPOCH, NUM_EPOCHS + 1):
    start = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion)
    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    elapsed = time.time() - start
    print(f'  Epoch {epoch:>3}/{NUM_EPOCHS} | '
          f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
          f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f} | '
          f'Time: {elapsed:.1f}s')

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_weights  = copy.deepcopy(model.state_dict())
        torch.save(best_weights, CHECKPOINT)
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}.')
            break

    # Save resumable checkpoint after every epoch
    torch.save({
        'epoch'          : epoch,
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_val_loss'  : best_val_loss,
    }, RESUME_PATH)

    with open(HISTORY_PATH, 'w') as f:
        json.dump(history, f)

model.load_state_dict(best_weights)
print(f'\nBest val loss : {best_val_loss:.4f}')
print(f'Model saved to : {CHECKPOINT}')

In [ ]:
# ── Cell 9: Plot Training Curves ──────────────────────────────────────────────

epochs_ran = range(1, len(history['train_loss']) + 1)
warmup_end = 5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('EfficientNet-B0 — Training Curves', fontsize=14)

axes[0].plot(epochs_ran, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs_ran, history['val_loss'],   label='Val Loss',   linewidth=2)
axes[0].axvline(x=warmup_end, color='gray', linestyle='--', linewidth=1, label='Warmup end')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(epochs_ran, history['val_acc'],   label='Val Acc',   linewidth=2)
axes[1].axvline(x=warmup_end, color='gray', linestyle='--', linewidth=1, label='Warmup end')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
save_path = FIGURES_DIR / '04_efficientnet_training_curves.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {save_path}')

In [ ]:
# ── Cell 10: Test Set Evaluation ──────────────────────────────────────────────

model.eval()
all_labels, all_preds, all_probs = [], [], []
softmax = nn.Softmax(dim=1)

with torch.no_grad():
    for images, labels in test_loader:
        images  = images.to(DEVICE)
        outputs = model(images)
        probs   = softmax(outputs).cpu().numpy()
        preds   = outputs.argmax(1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_prob = np.array(all_probs)

accuracy  = accuracy_score(y_true, y_pred)
macro_f1  = f1_score(y_true, y_pred, average='macro')
y_bin     = label_binarize(y_true, classes=list(range(5)))
macro_auc = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')

cm = confusion_matrix(y_true, y_pred)
per_class = {}
for i, name in enumerate(CLASS_NAMES):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    per_class[name] = {
        'sensitivity': round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0,
        'specificity': round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0,
    }

print('=' * 55)
print('  EFFICIENTNET-B0 — TEST SET RESULTS')
print('=' * 55)
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  Macro F1  : {macro_f1:.4f}')
print(f'  Macro AUC : {macro_auc:.4f}')
print('=' * 55)
for name, vals in per_class.items():
    print(f'  {name:<15} Sens: {vals["sensitivity"]:.4f}  Spec: {vals["specificity"]:.4f}')
print('=' * 55)

In [ ]:
# ── Cell 11: Confusion Matrix ─────────────────────────────────────────────────

cm_norm = confusion_matrix(y_true, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('EfficientNet-B0 — Normalised Confusion Matrix', fontsize=13)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
save_path = FIGURES_DIR / '04_efficientnet_confusion_matrix.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {save_path}')

In [ ]:
# ── Cell 12: ROC Curves ───────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.tab10.colors
for i, name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{name} (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('EfficientNet-B0 — Per-Class ROC Curves')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_path = FIGURES_DIR / '04_efficientnet_roc_curves.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {save_path}')

In [ ]:
# ── Cell 13: Overfitting Analysis and Save Results ────────────────────────────

final_train_acc = history['train_acc'][-1]
gap = final_train_acc - accuracy

print('=' * 55)
print('  EFFICIENTNET-B0 — OVERFITTING ANALYSIS')
print('=' * 55)
print(f'  Final Train Accuracy : {final_train_acc:.4f}')
print(f'  Test Accuracy        : {accuracy:.4f}')
print(f'  Gap                  : {gap:.4f}')
print(f'  Status               : {"WARNING — gap > 5%" if gap > 0.05 else "OK"}')
print('=' * 55)

results = {
    'model'     : 'EfficientNet-B0',
    'accuracy'  : round(accuracy, 4),
    'macro_f1'  : round(macro_f1, 4),
    'macro_auc' : round(macro_auc, 4),
    'train_acc' : round(final_train_acc, 4),
    'gap'       : round(gap, 4),
    'per_class' : per_class,
}
with open(MODELS_DIR / 'efficientnet_b0_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved to Drive.')

In [ ]:
# ── Cell 14: Save Notebook to Drive ──────────────────────────────────────────

try:
    shutil.copy('/content/04_efficientnet_b0.ipynb',
                str(BASE_DIR / 'notebooks' / '04_efficientnet_b0.ipynb'))
    print('Notebook saved to Drive.')
except:
    print('Use File > Save a copy in Drive to save this notebook.')